# seq_325030 barcode subsampling

seq_325030 carried the most barcodes (10,933) in sublibrary L3a2, exceeding MPRAnalyze's capacity, so ~half (5,486) were randomly subsampled while retaining well-above-average coverage.

In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np
import pybedtools
from pybedtools import BedTool
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import itertools
import re
import ast
import math

# --- Shared helpers: const.py lives in analyses/common ---
NB_DIR = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(NB_DIR, '..', 'common')))
import const
from const import pos_active_ctrl_color, neg_active_ctrl_color, highlight_color, custom_cmap
from const import set_equal_plot_limits, plot_color_pallete, custom_cmap_bolder, FONT_SIZE_small
const.set_plot_style()

# --- Paths (EDIT): inputs read / outputs written by this notebook ---
COMB_DF_CSV = '/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes/quantitative_analysis_combined/comb_df_combined_fdr.csv'  # combined per-oligo table with FDR
output_path = '.'  # where plots/tables are written

In [ ]:
comb_df = pd.read_csv(COMB_DF_CSV)
comaprative_input = comb_df[comb_df['input_comparative_combined']=='yes']

In [ ]:
# seq_325030 oligos in the full dataset
seq_325030_rows = comb_df[comb_df['oligo'].str.contains('seq_325030', na=False)]
print("seq_325030 oligos (in full dataset):")
print(seq_325030_rows[['oligo', 'count_rep_comb']].to_string())
print(f"\nCount values for seq_325030: {seq_325030_rows['count_rep_comb'].values}")

In [ ]:
# Extract library information from oligo names
def extract_library(oligo_name):
    """Extract library from oligo name (e.g., 'a2_L3' from end of string)"""
    if pd.isna(oligo_name):
        return None
    # Match library pattern at the end: aX_LY (e.g., a2_L3)
    match = re.search(r'(a\d+_L\d+)(?:$|_)', oligo_name)
    if match:
        return match.group(1)
    return None

# Test the function first
test_oligo = seq_325030_rows.iloc[0]['oligo']
test_lib = extract_library(test_oligo)
print(f"Test extraction on '{test_oligo}':")
print(f"  Extracted library: '{test_lib}'")

# Now apply to both dataframes
comb_df['library'] = comb_df['oligo'].apply(extract_library)
comaprative_input = comaprative_input.copy()  # Avoid the SettingWithCopyWarning
comaprative_input['library'] = comaprative_input['oligo'].apply(extract_library)

# Get seq_325030 from comparative input
seq_325030_rows = comaprative_input[comaprative_input['oligo'].str.contains('seq_325030', na=False)]

print(f"\n{'='*80}")
print("seq_325030 oligos in comparative input:")
print(f"{'='*80}")
print(seq_325030_rows[['oligo', 'count_rep_comb', 'library']].to_string())

seq_325030_counts = seq_325030_rows['count_rep_comb'].dropna().values
seq_325030_library = seq_325030_rows['library'].iloc[0]

print(f"\nseq_325030 counts: {seq_325030_counts}")
print(f"seq_325030 library: {seq_325030_library}")

# Calculate quantiles against all sequences in comparative input
all_counts_comp = comaprative_input['count_rep_comb'].dropna()
print(f"\n{'='*80}")
print(f"Quantiles vs ALL sequences in comparative dataset (n={len(all_counts_comp)}):")
print(f"{'='*80}")
for idx, row in seq_325030_rows.iterrows():
    count = row['count_rep_comb']
    oligo_short = row['oligo'].replace('seq_325030_chr14:90968484-90968753_SCREEN_', '')
    quantile = stats.percentileofscore(all_counts_comp, count)
    print(f"  {oligo_short}: {count:.0f} barcodes -> {quantile:.4f}th percentile")

# Get counts for the same library
if seq_325030_library:
    same_library_data = comaprative_input[comaprative_input['library'] == seq_325030_library]['count_rep_comb'].dropna()
    print(f"\n{'='*80}")
    print(f"Quantiles vs sequences in {seq_325030_library} library (n={len(same_library_data)}):")
    print(f"{'='*80}")
    for idx, row in seq_325030_rows.iterrows():
        count = row['count_rep_comb']
        oligo_short = row['oligo'].replace('seq_325030_chr14:90968484-90968753_SCREEN_', '')
        quantile = stats.percentileofscore(same_library_data, count)
        print(f"  {oligo_short}: {count:.0f} barcodes -> {quantile:.4f}th percentile")

In [ ]:
# Create a visualization with seq_325030 highlighted
fig, ax = plt.subplots(figsize=(9, 5))

all_data = comaprative_input['count_rep_comb'].dropna()

# Plot all data
ax.hist(all_data, bins=100, color=plot_color_pallete['barcode'], edgecolor='white', alpha=0.6, label='All other sequences')

# Highlight seq_325030
for count in seq_325030_counts:
    ax.axvline(count, color=highlight_color, linestyle='-', linewidth=2.5, alpha=0.9)

# Add mean and median lines
ax.axvline(all_data.mean(), color='black', linestyle='--', linewidth=2, label=f"Mean: {all_data.mean():.0f}", alpha=0.7)
ax.axvline(all_data.median(), color='gray', linestyle='--', linewidth=1.5, label=f"Median: {all_data.median():.0f}", alpha=0.7)

ax.set_yscale('log')
ax.set_xlabel('count_rep_comb (barcode count)', fontsize=FONT_SIZE_small)
ax.set_ylabel('Count (log scale)', fontsize=FONT_SIZE_small)
ax.set_title('Distribution of Barcode Counts\nwith seq_325030 Highlighted', fontsize=FONT_SIZE_small)
ax.legend(frameon=False, fontsize=FONT_SIZE_small-2)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(output_path + 'seq_325030_barcode_analysis.pdf', bbox_inches='tight', dpi=300)
plt.savefig(output_path + 'seq_325030_barcode_analysis.png', bbox_inches='tight', dpi=300)
plt.show()

print(f"\nPlots saved to {output_path}")